In [30]:
from libraries import *
from logger import setup_logger

import torch
from torch.utils.data import Dataset, DataLoader

import bisect
from itertools import accumulate
import argparse

import sys
import os
import logger

import torch.nn as nn

import torch
from torch import optim

from Utils import setup_lr_scheduler, setup_optimizer, save_checkpoint, load_checkpoint

In [31]:
def train_model_CVAE(trainloader, model, optimizer, loss_fn, single_batch=False):
    '''Method for training model (updating model params) based on given criterion'''
    
    model.train()

    total_loss = 0
    total_samples = 0

    for sample in trainloader:
        model.zero_grad()
        input = sample['y'].cuda() if next(model.parameters()).is_cuda else sample['y']
        condition = sample['X'].cuda() if next(model.parameters()).is_cuda else sample['X']

        output = model(input, condition)
        loss, BCE, KLD, l1Loss = loss_fn(*output)
        batch_size = len(input)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()*batch_size
        total_samples += batch_size

        if single_batch:
            break

    return {'train_loss': total_loss / total_samples, "BCE": BCE, "KLD":KLD, "l1Loss":l1Loss }


In [32]:
class ScreenDataset(Dataset):
    def __init__(self, h5adfile):
        self.data = sc.read(h5adfile)
        self.covariates = self.data.uns["covariates"]
        print(self.covariates)
        self.X = self.data.obs[ self.covariates ].to_numpy()
        self.y = self.data.X


    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        #y = self.data.X[idx:idx+1]
        y = self.y[idx]
        X = self.X[idx]
        return {'y': torch.from_numpy(y).float(), 'X': torch.from_numpy(X).float()}
    
    
    def get_genes(self):
        """ Return list of genes """
        return list(self.data.var_names)

    def get_targets(self):
        """ Return list of ko targets """
        return list(self.covariates)  
    

    
class VAE(nn.Module):

    def __init__(self, n_inputs, n_latents, n_cond, n_cond_in):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_latents = n_latents
        self.n_cond = n_cond

        self.encoder = Encoder(n_inputs=n_inputs, n_latents=n_latents, n_cond=n_cond)
        self.decoder = Decoder(n_inputs=n_inputs, n_latents=n_latents, n_cond=n_cond)
        self.embedding = EmbeddingLayer(n_in=n_cond_in, n_out=n_cond)

    def forward(self, x, c):

        
        c = self.embedding(c)

        means, log_var = self.encoder(x, c)
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        z = eps * std + means

        print("c decoder: ")
        print(c)

        recon_x = self.decoder(z, c)

        return recon_x, x, means, log_var, c

    def inference(self, x, c):
        c = self.embedding(c)
        means, _ = self.encoder(x, c)
        return means

    def generate(self, z, c):
        c = self.embedding(c)
        recon_x = self.decoder(z, c)
        return recon_x

    def get_embedding(self, c):
        return self.embedding(c)


    
class Encoder(nn.Module):
    def __init__(self, n_inputs, n_latents, n_cond):
        super().__init__()

        self.encoder = nn.Sequential(nn.Linear(n_inputs+n_cond, 512),
                                     nn.BatchNorm1d(512),
                                     nn.ReLU(inplace=True),
                                     nn.Linear(512, 512),
                                    )

        self.linear_means = nn.Linear(512, n_latents)
        self.linear_log_var = nn.Linear(512, n_latents)

    def forward(self, x, c):
        
        
        x = torch.cat((x, c), dim=1)
        x = self.encoder(x)
        means = self.linear_means(x)
        logvar = self.linear_log_var(x)
        return means, logvar

    
class Decoder(nn.Module):
    def __init__(self, n_inputs, n_latents, n_cond):
        super().__init__()

        self.decoder = nn.Sequential(nn.Linear(n_latents + n_cond, 512),
                                     nn.ReLU(inplace=True),
                                     nn.Linear(512, n_inputs),
                                    )

    def forward(self, z, c):
        x = torch.cat((z, c), dim=1)
        x = self.decoder(x)
        return x

        
def get_loss_fn(alpha, beta):

    def vae_loss_fn(recon_x, x, mean, log_var, c):
        BCE = torch.nn.functional.mse_loss(
            recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
        
        l1Loss = torch.sum(torch.abs(c))

        return (BCE + alpha*KLD)/x.size(0), BCE, KLD, l1Loss

    return vae_loss_fn


class EmbeddingLayer(nn.Module):
    def __init__(self, n_in, n_out):
        super(EmbeddingLayer, self).__init__()
        self.embedding_layer = nn.Sequential(nn.Linear(n_in, n_out))
        self.mysoftmax = nn.Softmax(dim=-1)
        
    def gumbel_softmax(self, embeds, tau=0.5, hard=False):
        
        gumbels = -torch.log(-torch.log(torch.rand_like(embeds) + 1e-20) + 1e-20)
        y = embeds + gumbels
        y_soft = torch.sigmoid(y / tau)

        #y_soft = F.sigmoid(y / tau)
        if hard:
            return (y_soft == y_soft.max(dim=-1, keepdim=True)[0]).float()
        else:
            return y_soft


    def forward(self, x):
        print("c initial")
        print(x)
        
        y_emb = self.embedding_layer(x)
#         print("y_emb size:")
#         print(y_emb.size())
        
#         print(y_emb)
        
#         print("y_emb sum: ")
#         print(torch.sum(y_emb, dim=1))
         
        print("c embedded")
        print(y_emb)
        y_emb_gumbel = self.gumbel_softmax(y_emb, hard=False)
        
        print("c embedded gumbel")
        print(y_emb_gumbel)
        
#         print("y_emb_gumbel size: ")
#         print(y_emb_gumbel.size())
        
#         print(y_emb_gumbel)
#         print("y_emb_gumbel sum: ")
#         print(torch.sum(y_emb_gumbel, dim=1))
        
        
        return y_emb_gumbel



In [33]:
trainfile="/home/eraslab1/Projects/E3Ligase/analysisSingle/ComboData_v2/outputs/anndata/adataKOSingles_leidenRegOut_res01_train.h5ad"
controlfile="/home/eraslab1/Projects/E3Ligase/analysisSingle/ComboData_v2/outputs/anndata/adataControl.h5ad"
valfile="/home/eraslab1/Projects/E3Ligase/analysisSingle/ComboData_v2/outputs/anndata/adataKOSingles_leidenRegOut_res01_test.h5ad"
save_dir='/home/eraslab1/Projects/E3Ligase/analysisSingle/ComboData_v2/outputs/Models/'
model_dir="model_alpha_40_KOSingles_leidenRegOut_res01_gumbel_v2_tau2"
seed=42
n_cond_in=14
n_inputs=1011
n_cond=64
n_latents=64
model_type="CVAE"
optimizer='adam'
scheduler='none'
alpha=40
beta=0
theta=0
batch_size=100
num_workers=40
lr=0.0001
max_epochs=2
weight_decay=0
use_gpu=True
project_dir="/home/eraslab1/Projects/E3Ligase/analysisSingle/ComboData_v2/"
gpu_number='0'


In [34]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]='0'

torch.cuda.set_device(0)


In [35]:
trainset = ScreenDataset(trainfile)
trainloader = DataLoader(trainset,
                         batch_size=batch_size,
                     drop_last=True,
                     shuffle=True, 
                     num_workers=num_workers)
    


['Ambra1' 'Ankfy1' 'Crebbp' 'Ep300' 'Fbxw7' 'Klhl6' 'March6' 'Pparg'
 'Rfwd2' 'Socs3' 'Syvn1' 'Traf2' 'Trip12' 'Wdr82']


In [36]:
net = VAE(n_inputs=n_inputs,
          n_latents=n_latents,
          n_cond=n_cond,
          n_cond_in=n_cond_in)

optimizer = setup_optimizer(name=optimizer, param_list=[{'params': net.parameters(), 
                                                          'lr': lr,
                                                          'weight_decay': weight_decay}])
if use_gpu:
    net.cuda()


In [37]:
for epoch in range(2):

    loss_fn = get_loss_fn(alpha, beta)
    train_summary = train_model_CVAE(trainloader=trainloader, 
                                model=net, 
                                optimizer=optimizer, loss_fn=loss_fn)
 

c initial
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]], device='cuda:0')
c embedded
tensor([[ 0.2588, -0.4597, -0.0888,  ..., -0.0551, -0.3910,  0.3199],
        [ 0.1838, -0.0756, -0.1934,  ..., -0.1931, -0.3083,  0.0392],
        [ 0.0438, -0.4557,  0.1544,  ..., -0.1098, -0.3396,  0.3782],
        ...,
        [-0.0073, -0.3988, -0.2025,  ..., -0.4469, -0.3006,  0.0393],
        [ 0.2960, -0.2662, -0.1243,  ..., -0.5128, -0.3405,  0.1191],
        [-0.1477, -0.2369, -0.1116,  ..., -0.4480, -0.0473,  0.2232]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
c embedded gumbel
tensor([[0.0820, 0.9369, 0.3163,  ..., 0.8187, 0.3185, 0.3992],
        [0.8968, 0.8178, 0.8075,  ..., 0.7969, 0.4281, 0.6371],
        [0.4641, 0.9565, 0.1257,  ..., 0.2474, 0.9466, 0.6585],
        ...,
 

In [ ]:
train_summary